# 02. 트랙으로 — 준비된 연주를 분리된 stem으로

이 노트북은 **준비된 합주/협주곡 음원**을 AI로 분리해, 다시 조합할 수 있는 **stem (분리된 트랙)** 으로 바꾸는 데모입니다.
수업에서는 아래의 예시 음원을 사용해 안정적으로 시연합니다.

**도구**: [Demucs](https://github.com/facebookresearch/demucs) — Meta가 만든 오픈소스 소스 분리 AI

▶ Colab에서 실행한다면 아래 설치 셀부터 시작하세요.
서버 시연 환경에서 이미 `setup.sh`를 실행했다면, 설치 셀은 건너뛰고 다음 셀로 바로 넘어가도 됩니다.


In [ ]:
# Colab용 설치 (서버 시연 환경에서는 보통 건너뛰어도 됩니다)
!pip install -q demucs librosa soundfile matplotlib

import warnings
warnings.filterwarnings('ignore')

print('✅ 설치 완료!')

---
## 1. 준비된 예시 음원 선택

수업에서는 먼저 **이미 준비된 합주/협주곡 예시 음원**으로 데모를 진행합니다.
아래 셀은 먼저 repo의 `assets/` 폴더를 확인하고, 없으면 예시 파일을 다운로드합니다.


In [ ]:
import os
import urllib.request
from pathlib import Path

REPO_URL = 'https://raw.githubusercontent.com/joonhyungbae/pianokit/main/assets'
ASSETS_DIR = Path('assets')

candidate_paths = [ASSETS_DIR / 'concerto_example.wav', Path('concerto_example.wav')]
for candidate in candidate_paths:
    if candidate.exists():
        input_audio = str(candidate)
        print(f'✅ 예시 음원 사용: {input_audio}')
        break
else:
    input_audio = 'concerto_example.wav'
    urllib.request.urlretrieve(f'{REPO_URL}/concerto_example.wav', input_audio)
    print(f'✅ 다운로드 완료: {input_audio}')


---
## 2. AI로 악기별 분리하기

Demucs는 음원의 파형 패턴을 분석해서, 각 악기가 어떤 소리를 내고 있는지 구분합니다.
하나의 음원에서 **보컬, 드럼, 베이스, 기타/피아노** 같은 stem을 분리해, 다시 쓸 수 있는 음향 재료로 바꿉니다.

⏰ 1~2분 정도 걸립니다. ▶ 아래 셀을 실행하세요.

In [ ]:
import subprocess

print('🔄 AI가 악기를 분리하고 있습니다... (1~2분 소요)')
print('   htdemucs_ft (fine-tuned) 모델 사용 — Demucs v4 최고 품질')

# htdemucs_ft: fine-tuned 모델 (4-stem: vocals, drums, bass, other)
result = subprocess.run(
    ['python', '-m', 'demucs', '-n', 'htdemucs_ft', input_audio],
    capture_output=True, text=True
)

# Demucs는 separated/htdemucs_ft/{파일명}/ 디렉토리에 결과 저장
stem_name = os.path.splitext(os.path.basename(input_audio))[0]
output_dir = f'separated/htdemucs_ft/{stem_name}'

if os.path.exists(output_dir):
    tracks = os.listdir(output_dir)
    print(f'✅ 분리 완료! {len(tracks)}개 트랙:')
    for t in sorted(tracks):
        print(f'  🎵 {t}')
else:
    print('⚠️ 분리에 실패했습니다.')
    print(result.stderr[-500:] if result.stderr else '에러 정보 없음')

---
## 3. 분리된 트랙 들어보기

아래에서 각 트랙을 따로 들어볼 수 있습니다.
원본과 번갈아 들으면서 무엇이 남고 무엇이 분리되었는지 확인해보세요.
이 단계에서 연주는 다시 조합하고 편곡할 수 있는 **음향 트랙**이 됩니다.

In [ ]:
import librosa
import soundfile as sf
import matplotlib.pyplot as plt
import numpy as np
import IPython.display as ipd

track_labels = {
    'vocals.wav': '🎤 보컬',
    'drums.wav': '🥁 드럼',
    'bass.wav': '🎸 베이스',
    'other.wav': '🎹 기타/피아노',
    'no_vocals.wav': '🎶 보컬 제거 (반주)',
}

if os.path.exists(output_dir):
    tracks = sorted(os.listdir(output_dir))
    fig, axes = plt.subplots(len(tracks), 1, figsize=(14, 3 * len(tracks)))
    if len(tracks) == 1:
        axes = [axes]

    for idx, track_file in enumerate(tracks):
        track_path = os.path.join(output_dir, track_file)
        label = track_labels.get(track_file, track_file)

        # 파형 시각화
        y, sr = librosa.load(track_path, sr=None, mono=True)
        times = np.arange(len(y)) / sr
        axes[idx].plot(times, y, linewidth=0.3, color='#4FC3F7')
        axes[idx].set_title(label, fontsize=12)
        axes[idx].set_xlim(0, times[-1])
        axes[idx].set_ylabel('진폭')

        # 재생 위젯
        print(f'\n{label}:')
        ipd.display(ipd.Audio(track_path))

    axes[-1].set_xlabel('시간 (초)')
    plt.tight_layout()
    plt.show()
else:
    print('⚠️ 분리된 트랙을 찾을 수 없습니다. 3번 셀을 먼저 실행해주세요.')

---
## 4. 결과 저장

▶ 아래 셀은 분리된 트랙이 저장된 위치를 보여주고, Colab이라면 zip으로 묶어 바로 다운로드합니다.

💡 분리한 트랙은 이후 `무대화` 단계에서 레이어를 나누어 생각하거나, 공연용 재료를 설계할 때 참고할 수 있습니다.


In [ ]:
import zipfile
from pathlib import Path

output_path = Path(output_dir)
if output_path.exists():
    print(f'✅ 분리된 트랙 폴더: {output_path.resolve()}')
    try:
        from google.colab import files
        zip_name = f'{stem_name}_separated.zip'
        with zipfile.ZipFile(zip_name, 'w') as zf:
            for track_file in output_path.iterdir():
                zf.write(track_file, track_file.name)
        print('📦 Colab 다운로드용 zip 생성 완료')
        files.download(zip_name)
    except Exception:
        print('ℹ️ 서버/로컬 환경에서는 위 폴더의 파일을 그대로 사용하면 됩니다.')
else:
    print('⚠️ 분리된 트랙이 없습니다.')
